# 06 — Inference Optimizasyonu

**İlan maddesi:** *"Çıkarım (Inference) optimizasyonu (niceleme/quantization,
budama/pruning, bilgi damıtma/distillation) gerçekleştirerek modellerin gerçek zamanlı
ve kısıtlı donanım ortamlarında kullanılabilirliğini sağlamak."*

Üç tekniği de deneyip, `src/optimize/benchmark.py` ile gecikme/bellek üzerindeki
etkilerini ölçeceğiz.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) hiçbir şey yapmadan devam eder.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"

if not os.path.exists(PROJECT_DIR):
    try:
        from google.colab import drive
        # drive.mount() zaten mount edilmişse anında geri döner (idempotent);
        # os.path.exists("/content/drive") ile "mount edilmiş mi" kontrol etmek
        # güvenilmez çünkü klasör, başarısız/yarım bir mount denemesinden sonra
        # bile var olabilir. Bu yüzden koşulsuz çağırıyoruz.
        drive.mount("/content/drive", force_remount=True)
        if os.path.exists(DRIVE_ZIP_PATH):
            import shutil
            shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
        else:
            print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
                  "yükleyin ya da kendi reponuzu klonlayın: "
                  f"!git clone <repo-url> {PROJECT_DIR}")
    except ImportError:
        pass  # Colab dışında (yerelde) çalışıyorsanız bu adım gerekmez.

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)


## 1. Niceleme (Quantization)

4-bit ve dinamik INT8 (CPU) karşılaştırması.

In [ ]:
from src.optimize.quantize import load_4bit
from src.config import MODEL_CONFIG

model_4bit = load_4bit(MODEL_CONFIG.base_llm)
print(model_4bit.get_memory_footprint() / 1e6, "MB (4-bit)")


## 2. Budama (Pruning)

02. notebook'ta eğittiğiniz sınıflandırıcı üzerinde (önce `classification.train()` çalıştırmış olmanız gerekir).

In [ ]:
from transformers import AutoModelForSequenceClassification
from src.optimize.prune import magnitude_prune, sparsity_report
from src.config import MODELS_DIR

clf = AutoModelForSequenceClassification.from_pretrained(str(MODELS_DIR / "classifier"))
pruned_clf = magnitude_prune(clf, amount=0.3)

print("Öncesi:", sparsity_report(clf))
print("Sonrası:", sparsity_report(pruned_clf))


## 3. Bilgi Damıtma (Distillation)

Büyük sınıflandırıcıdan küçük bir DistilBERT öğrenciye bilgi aktarımı.

In [ ]:
from src.optimize.distill import distill

# texts/labels: 02. notebook'taki etiketli örnekleriniz
# student_dir = distill(texts, labels, epochs=3)
print("Örnek veri seti hazırladıktan sonra distill(texts, labels) çağırın.")


## 4. Benchmark karşılaştırması

In [ ]:
from src.optimize.benchmark import benchmark_inference, compare
import torch

def predict_fp32(x):
    with torch.no_grad():
        return clf(**x).logits

sample_input = clf.dummy_inputs if hasattr(clf, "dummy_inputs") else None
# Gerçek tokenized girdilerle değiştirin:
# result_fp32 = benchmark_inference(predict_fp32, [sample_input]*5, name="fp32")
# result_pruned = benchmark_inference(lambda x: pruned_clf(**x).logits, [sample_input]*5, name="pruned")
# print(compare([result_fp32, result_pruned]))
print("Yukarıdaki yorum satırlarını gerçek tokenized girdilerinizle aktif edin.")
